In [ ]:
!pip install boto3 python-dotenv marker-pdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.7/195.7 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.2/223.2 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 91.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 96.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 121.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22

In [ ]:
import os
import boto3
import subprocess
import shutil
from dotenv import load_dotenv

In [ ]:
# 1. Cargar variables de entorno
load_dotenv()

AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_SESSION_TOKEN = os.getenv("AWS_SESSION_TOKEN")
AWS_REGION = os.getenv("AWS_REGION")

S3_BUCKET_NAME = os.getenv("S3_BUCKET_NAME")
S3_PREFIX_DOCS = os.getenv("S3_PREFIX_DOCS")
S3_PREFIX_BRONCE= os.getenv("S3_PREFIX_BRONCE")

In [ ]:
# 2. Inicializar cliente S3
s3_client = boto3.client(
    's3',
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    aws_session_token=AWS_SESSION_TOKEN,
    region_name=AWS_REGION
)

In [ ]:
def obtener_archivos_s3(prefijo):
    """Devuelve una lista de nombres de archivos en un prefijo dado."""
    archivos = []
    paginator = s3_client.get_paginator('list_objects_v2')
    for page in paginator.paginate(Bucket=S3_BUCKET_NAME, Prefix=prefijo):
        if 'Contents' in page:
            for obj in page['Contents']:
                # Evitar agregar el propio directorio como archivo
                if not obj['Key'].endswith('/'):
                    archivos.append(obj['Key'])
    return archivos

In [ ]:
def obtener_carpetas_procesadas():
    """Obtiene los nombres de los documentos que ya tienen su carpeta en Bronce."""
    carpetas = set()
    paginator = s3_client.get_paginator('list_objects_v2')
    # Buscamos usando el delimitador '/' para encontrar "subcarpetas"
    for page in paginator.paginate(Bucket=S3_BUCKET_NAME, Prefix=S3_PREFIX_BRONCE, Delimiter='/'):
        if 'CommonPrefixes' in page:
            for prefix in page['CommonPrefixes']:
                # Extraemos solo el nombre de la carpeta (ej. de 'bronze/processed/doc1/' sacamos 'doc1')
                nombre_carpeta = prefix['Prefix'].replace(S3_PREFIX_BRONCE, '').strip('/')
                carpetas.add(nombre_carpeta)
    return carpetas

In [ ]:
def procesar_pipeline_ingesta():
    print("Iniciando Pipeline: Ingesta -> Bronce (Estructura de Cápsulas)")

    pdfs_en_docs = obtener_archivos_s3(S3_PREFIX_DOCS)
    carpetas_en_bronce = obtener_carpetas_procesadas()

    for s3_key in pdfs_en_docs:
        nombre_pdf = os.path.basename(s3_key)
        nombre_base = nombre_pdf.replace('.pdf', '')

        # VERIFICADOR: Si la carpeta ya existe en Bronce, lo saltamos
        if nombre_base in carpetas_en_bronce:
            print(f" Saltando: {nombre_pdf} (Carpeta ya existe en Bronce)")
            continue

        print(f"\n Descargando: {nombre_pdf}...")
        local_pdf_path = f"/content/{nombre_pdf}"
        s3_client.download_file(S3_BUCKET_NAME, s3_key, local_pdf_path)

        local_out_dir = f"/content/out_{nombre_base}"
        os.makedirs(local_out_dir, exist_ok=True)

        print(f"Procesando con Marker: {nombre_pdf}...")
        comando = ["marker_single", local_pdf_path, "--output_dir", local_out_dir]
        resultado = subprocess.run(comando, capture_output=True, text=True)

        if resultado.returncode == 0:
            # Marker crea los archivos dentro de esta subcarpeta local
            carpeta_marker = os.path.join(local_out_dir, nombre_base)

            if os.path.exists(carpeta_marker):
                archivos_subidos = 0

                # Iteramos sobre todo lo que generó Marker (el .md y las imágenes)
                for archivo in os.listdir(carpeta_marker):
                    ruta_local_archivo = os.path.join(carpeta_marker, archivo)

                    if os.path.isfile(ruta_local_archivo):
                        # La nueva ruta en S3 agrupa todo bajo el nombre del documento
                        ruta_s3_destino = f"{S3_PREFIX_BRONCE}{nombre_base}/{archivo}"
                        s3_client.upload_file(ruta_local_archivo, S3_BUCKET_NAME, ruta_s3_destino)
                        archivos_subidos += 1

                print(f"Éxito. Se creó la cápsula '{nombre_base}/' con {archivos_subidos} archivos.")
            else:
                print(f"Error: Marker no generó la carpeta de salida para {nombre_pdf}")
        else:
            print(f"Error de Marker procesando {nombre_pdf}:")
            print(resultado.stderr)

        # Limpieza local para no saturar la máquina
        if os.path.exists(local_pdf_path): os.remove(local_pdf_path)
        if os.path.exists(local_out_dir): shutil.rmtree(local_out_dir)

In [11]:
# Ejecutar el flujo
procesar_pipeline_ingesta()

Iniciando Pipeline: Ingesta -> Bronce (Estructura de Cápsulas)

 Descargando: Esmalte Uretano AR Comp. B.pdf...
Procesando con Marker: Esmalte Uretano AR Comp. B.pdf...
Éxito. Se creó la cápsula 'Esmalte Uretano AR Comp. B/' con 4 archivos.

 Descargando: FDS 1 -pintura-antideslizante-para-canchas.pdf...
Procesando con Marker: FDS 1 -pintura-antideslizante-para-canchas.pdf...
Éxito. Se creó la cápsula 'FDS 1 -pintura-antideslizante-para-canchas/' con 25 archivos.

 Descargando: FDS 10 - pintura-epoxi-poliamida-blanco-13243-10012925-10344427-hoja-de-seguridad-pdf.pdf...
Procesando con Marker: FDS 10 - pintura-epoxi-poliamida-blanco-13243-10012925-10344427-hoja-de-seguridad-pdf.pdf...
Éxito. Se creó la cápsula 'FDS 10 - pintura-epoxi-poliamida-blanco-13243-10012925-10344427-hoja-de-seguridad-pdf/' con 22 archivos.

 Descargando: FDS 11  - Ecosphere Premium.pdf...
Procesando con Marker: FDS 11  - Ecosphere Premium.pdf...
Éxito. Se creó la cápsula 'FDS 11  - Ecosphere Premium/' con 26 arch